# NZ Meal Cost Optimizer

Find the cheapest Pak'nSave for your recipe.

1. Enter your address
2. Enter a dish name
3. Compare prices across nearby stores

In [ ]:
import cloudscraper
import pandas as pd
import requests
import math
from pathlib import Path

stores = pd.read_csv("../data/paknsave_stores.csv")
print(f"Loaded {len(stores)} Pak'nSave stores")

In [ ]:
BASE = "https://api-prod.prod.fsniwaikato.kiwi/prod"

class PaknSaveAPI:
    def __init__(self):
        self.scraper = cloudscraper.create_scraper()
        self._token = None

    def _ensure_token(self):
        if self._token:
            return
        r = self.scraper.post(
            f"{BASE}/mobile/user/login/guest",
            json={"banner": "PNS"},
            headers={"User-Agent": "PAKnSAVEApp/4.32.0", "Content-Type": "application/json"},
        )
        r.raise_for_status()
        self._token = r.json()["access_token"]
        self._auth = {
            "Authorization": f"Bearer {self._token}",
            "access_token": self._token,
            "User-Agent": "PAKnSAVEApp/4.32.0",
            "Content-Type": "application/json",
        }

    def search_products(self, store_id: str, query: str):
        self._ensure_token()
        r = self.scraper.post(
            f"{BASE}/mobile/ecomm-products/PNS/{store_id}/search?q={query}",
            headers=self._auth, json=[],
        )
        if r.status_code == 200:
            return r.json()
        return None

api = PaknSaveAPI()
print("API client ready")

In [ ]:
def geocode(address):
    r = requests.get(
        "https://nominatim.openstreetmap.org/search",
        headers={"User-Agent": "NZMealCostOptimizer/1.0"},
        params={"q": address, "format": "json", "limit": 1},
    )
    if r.status_code == 200 and r.json():
        loc = r.json()[0]
        return float(loc["lat"]), float(loc["lon"])
    return None, None

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat / 2) ** 2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2) ** 2
    return R * 2 * math.asin(math.sqrt(a))

def find_nearby(user_lat, user_lon, radius_km=5):
    df = stores.copy()
    df["distance_km"] = df.apply(
        lambda r: haversine(user_lat, user_lon, r["latitude"], r["longitude"]), axis=1,
    )
    return df[df["distance_km"] <= radius_km].sort_values("distance_km")

def derive_unit_label(units):
    """Convert API units (e.g. '400g', 'kg', '12pk') to a human-readable pricing label."""
    if not units:
        return "each"
    u = units.lower().strip()
    if u == "kg":
        return "/kg"
    if u == "l":
        return "/L"
    if u.endswith("g"):
        return f"each ({units})"
    if u.endswith("ml"):
        return f"each ({units})"
    if u.endswith("l") and u != "l":
        return f"each ({units})"
    if "pk" in u or "pack" in u:
        return f"per pack ({units})"
    return "each"

In [ ]:
DISH_INGREDIENTS = {
    "spaghetti bolognese": ["beef mince", "spaghetti pasta", "canned tomatoes", "onion", "carrot", "garlic", "mixed herbs"],
    "chicken stir fry": ["chicken breast", "stir fry vegetables", "soy sauce", "rice noodles"],
    "beef stir fry": ["beef strips", "stir fry vegetables", "soy sauce", "rice noodles"],
    "roast lamb": ["lamb roast", "potato", "carrot", "broccoli", "stock"],
    "chicken curry": ["chicken thigh", "curry paste", "coconut milk", "rice", "onion"],
    "beef curry": ["diced beef", "curry paste", "coconut milk", "rice", "onion"],
    "fish and chips": ["fish fillet", "potato", "oil"],
    "nachos": ["beef mince", "tortilla chips", "cheese", "beans", "sour cream"],
    "pumpkin soup": ["pumpkin", "onion", "cream", "stock", "bread"],
    "tacos": ["beef mince", "taco shells", "lettuce", "tomato", "cheese", "sour cream"],
    "lamb chops": ["lamb chops", "potato", "mint sauce"],
    "butter chicken": ["chicken thigh", "butter chicken sauce", "rice", "cream"],
    "lasagne": ["beef mince", "lasagne sheets", "cheese", "canned tomatoes", "milk", "butter", "flour"],
    "shepherd's pie": ["beef mince", "potato", "carrot", "peas", "stock"],
    "pizza": ["pizza base", "pizza sauce", "cheese", "pepperoni"],
    "vegie stir fry": ["stir fry vegetables", "tofu", "soy sauce", "rice noodles", "garlic"],
    "frittata": ["eggs", "potato", "onion", "cheese", "milk"],
    "pancakes": ["flour", "eggs", "milk", "sugar", "butter"],
    "chicken soup": ["chicken breast", "carrot", "onion", "celery", "stock", "pasta"],
    "tomato pasta": ["pasta", "canned tomatoes", "garlic", "olive oil", "mixed herbs", "cheese"],
    "chicken katsu": ["chicken breast", "flour", "eggs", "bread", "rice", "katsu sauce"],
}

DISH_QUANTITIES = {
    "spaghetti bolognese": {
        "beef mince": "500g",
        "spaghetti pasta": "400g",
        "canned tomatoes": "1 can (400g)",
        "onion": "1 medium",
        "carrot": "2 medium",
        "garlic": "2 cloves",
        "mixed herbs": "1 tsp",
    },
    "chicken stir fry": {
        "chicken breast": "2 fillets (~400g)",
        "stir fry vegetables": "1 bag (500g)",
        "soy sauce": "2 tbsp",
        "rice noodles": "250g",
    },
    "beef stir fry": {
        "beef strips": "400g",
        "stir fry vegetables": "1 bag (500g)",
        "soy sauce": "2 tbsp",
        "rice noodles": "250g",
    },
    "roast lamb": {
        "lamb roast": "1.2kg",
        "potato": "4 medium",
        "carrot": "3 medium",
        "broccoli": "1 head",
        "stock": "2 cups",
    },
    "chicken curry": {
        "chicken thigh": "500g",
        "curry paste": "2 tbsp",
        "coconut milk": "1 can (400ml)",
        "rice": "1.5 cups",
        "onion": "1 medium",
    },
    "beef curry": {
        "diced beef": "500g",
        "curry paste": "2 tbsp",
        "coconut milk": "1 can (400ml)",
        "rice": "1.5 cups",
        "onion": "1 medium",
    },
    "fish and chips": {
        "fish fillet": "2 fillets (~400g)",
        "potato": "4 medium",
        "oil": "for frying",
    },
    "nachos": {
        "beef mince": "300g",
        "tortilla chips": "1 bag (200g)",
        "cheese": "1 cup shredded",
        "beans": "1 can (400g)",
        "sour cream": "1/2 cup",
    },
    "pumpkin soup": {
        "pumpkin": "1kg",
        "onion": "1 medium",
        "cream": "1/2 cup",
        "stock": "2 cups",
        "bread": "4 slices",
    },
    "tacos": {
        "beef mince": "400g",
        "taco shells": "1 pack (12 shells)",
        "lettuce": "1/2 head",
        "tomato": "2 medium",
        "cheese": "1 cup shredded",
        "sour cream": "1/2 cup",
    },
    "lamb chops": {
        "lamb chops": "4 chops (~600g)",
        "potato": "4 medium",
        "mint sauce": "2 tbsp",
    },
    "butter chicken": {
        "chicken thigh": "500g",
        "butter chicken sauce": "1 jar",
        "rice": "1.5 cups",
        "cream": "1/2 cup",
    },
    "lasagne": {
        "beef mince": "500g",
        "lasagne sheets": "1 pack",
        "cheese": "1 cup shredded",
        "canned tomatoes": "1 can (400g)",
        "milk": "1 cup",
        "butter": "2 tbsp",
        "flour": "2 tbsp",
    },
    "shepherd's pie": {
        "beef mince": "500g",
        "potato": "4 medium",
        "carrot": "2 medium",
        "peas": "1 cup",
        "stock": "1/2 cup",
    },
    "pizza": {
        "pizza base": "1 base",
        "pizza sauce": "1/2 cup",
        "cheese": "1.5 cups shredded",
        "pepperoni": "1 pack",
    },
    "vegie stir fry": {
        "stir fry vegetables": "1 bag (500g)",
        "tofu": "1 block (400g)",
        "soy sauce": "2 tbsp",
        "rice noodles": "250g",
        "garlic": "2 cloves",
    },
    "frittata": {
        "eggs": "6 eggs",
        "potato": "2 medium",
        "onion": "1 medium",
        "cheese": "1 cup shredded",
        "milk": "1/4 cup",
    },
    "pancakes": {
        "flour": "1.5 cups",
        "eggs": "1 egg",
        "milk": "1 cup",
        "sugar": "2 tbsp",
        "butter": "2 tbsp",
    },
    "chicken soup": {
        "chicken breast": "2 fillets (~400g)",
        "carrot": "2 medium",
        "onion": "1 medium",
        "celery": "2 stalks",
        "stock": "4 cups",
        "pasta": "1 cup",
    },
    "tomato pasta": {
        "pasta": "400g",
        "canned tomatoes": "1 can (400g)",
        "garlic": "2 cloves",
        "olive oil": "2 tbsp",
        "mixed herbs": "1 tsp",
        "cheese": "1/4 cup grated",
    },
    "chicken katsu": {
        "chicken breast": "2 fillets (~400g)",
        "flour": "1/2 cup",
        "eggs": "2 eggs",
        "bread": "1 cup breadcrumbs",
        "rice": "1.5 cups",
        "katsu sauce": "1/3 cup",
    },
}

def get_ingredients(dish_name):
    return DISH_INGREDIENTS.get(dish_name.lower().strip(), [dish_name])

def get_quantities(dish_name):
    return DISH_QUANTITIES.get(dish_name.lower().strip(), {})

def search_ingredient(store_id, ingredient):
    results = api.search_products(store_id, ingredient)
    if not results:
        return None
    products = results.get("products", [])
    if not products:
        return None
    p = products[0]
    price_cents = p.get("price")
    if price_cents is None or price_cents <= 0:
        return None
    units = p.get("units", "")
    return {
        "name": p["name"],
        "brand": p.get("brand", ""),
        "price": price_cents / 100,
        "units": units,
        "unit_label": derive_unit_label(units),
    }

## Run the optimizer

Edit the two variables below and run this cell.

In [ ]:
# ═══════ YOUR INPUTS ═══════
USER_ADDRESS = "Botany Town Centre, Auckland 2013"
DISH_NAME = "frittata"
MAX_DISTANCE_KM = 5
# ═══════════════════════════

user_lat, user_lon = geocode(USER_ADDRESS)
if user_lat is None:
    raise SystemExit(f"Could not geocode: {USER_ADDRESS}")
print(f"Address: {USER_ADDRESS}")
print(f"Location: {user_lat:.5f}, {user_lon:.5f}\n")

nearby = find_nearby(user_lat, user_lon, MAX_DISTANCE_KM)
if nearby.empty:
    raise SystemExit(f"No stores within {MAX_DISTANCE_KM}km")

print(f"Nearby stores ({len(nearby)} within {MAX_DISTANCE_KM}km):")
for _, s in nearby.iterrows():
    print(f"  {s['name']:30s} {s['distance_km']:.2f} km")
print()

ingredients = get_ingredients(DISH_NAME)
quantities = get_quantities(DISH_NAME)
print(f"Dish: {DISH_NAME}")
print(f"Ingredients: {', '.join(ingredients)}\n")

all_results = []
for _, store in nearby.iterrows():
    store_id = store["store_id"]
    store_name = store["name"]
    print(f"--- {store_name} ---")
    total = 0.0
    for ing in ingredients:
        result = search_ingredient(store_id, ing)
        if result:
            qty = quantities.get(ing, "")
            qty_str = f" [{qty}]" if qty else ""
            all_results.append({**result, "store": store_name, "ingredient": ing, "distance_km": store["distance_km"]})
            print(f"  {ing:25s} ${result['price']:>5.2f}  {result['name']} ({result['units']}){qty_str}")
            total += result["price"]
        else:
            print(f"  {ing:25s}  NOT FOUND")
    print(f"  {'TOTAL':25s} ${total:.2f}\n")

if all_results:
    df = pd.DataFrame(all_results)
    summary = df.groupby("store").agg(
        total_cost=("price", "sum"),
        items_found=("ingredient", "count"),
        distance_km=("distance_km", "first"),
    ).sort_values("total_cost")
    print("=" * 65)
    print("COST COMPARISON")
    print("=" * 65)
    print(summary.to_string())
    print()
    best = summary.index[0]
    best_total = summary["total_cost"].iloc[0]
    print(f"Cheapest: {best} - ${best_total:.2f} total")

In [ ]:
# Per-store breakdown + best-case combo
if all_results:
    df = pd.DataFrame(all_results)
    quantities = get_quantities(DISH_NAME)

    # Build display table: each cell = "ProductName: $X.XX (unit_label)"
    store_names = sorted(df["store"].unique())
    ingredient_order = get_ingredients(DISH_NAME)

    rows = []
    for ing in ingredient_order:
        row = {"Ingredient": ing, "Qty": quantities.get(ing, "-")}
        for sn in store_names:
            match = df[(df["ingredient"] == ing) & (df["store"] == sn)]
            if not match.empty:
                r = match.iloc[0]
                row[sn] = f"{r['name']}: ${r['price']:.2f} ({r['unit_label']})"
            else:
                row[sn] = "NOT FOUND"

        # Best price across stores
        prices = []
        for sn in store_names:
            match = df[(df["ingredient"] == ing) & (df["store"] == sn)]
            if not match.empty:
                prices.append((sn, match.iloc[0]["price"]))
        if prices:
            best_sn, best_px = min(prices, key=lambda x: x[1])
            row["Best Price"] = f"${best_px:.2f}"
            row["Best Store"] = best_sn
        else:
            row["Best Price"] = "-"
            row["Best Store"] = "-"
        rows.append(row)

    table = pd.DataFrame(rows).set_index("Ingredient")

    # Totals row
    totals = {"Qty": ""}
    for sn in store_names:
        store_total = df[df["store"] == sn]["price"].sum()
        totals[sn] = f"${store_total:.2f}"
    best_total = sum(
        min(
            df[(df["ingredient"] == ing)]["price"].min()
            for ing in ingredient_order
            if not df[(df["ingredient"] == ing)].empty
        )
    )
    totals["Best Price"] = f"${best_total:.2f}"
    totals["Best Store"] = "(mix)"
    table.loc["TOTAL"] = totals

    print(f"{DISH_NAME.title()} — {len(ingredient_order)} ingredients across {len(store_names)} stores")
    print(f"Quantities shown are for a standard batch.\n")
    display(table)

    # Summary
    store_totals = df.groupby("store")["price"].sum().sort_values()
    cheapest_store = store_totals.index[0]
    cheapest_total = store_totals.iloc[0]
    
    print(f"\n{'='*65}")
    print(f"Cheapest single-store shop:  {cheapest_store:25s} ${cheapest_total:.2f}")
    print(f"Best case (mix stores):      {'—':25s} ${best_total:.2f}")
    print(f"Savings by mixing stores:    {'—':25s} ${cheapest_total - best_total:.2f}")
    print(f"{'='*65}")